### 05-01 Deployment Intro

Build model
<br>
Use flask framework to create a web service
<br>
Isolate the dependencies for the service so that it does not interfere with other services => Create a special environment for Python dependencies => Use pipenv
<br>
Add another layer for system dependencies => Use docker
<br>
Deploy container to cloud => Use AWS Elastic Beanstalk

### 05-02 Pickle

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold

from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

In [2]:
df = pd.read_csv('data-week-3.csv')

df.columns = df.columns.str.lower().str.replace(' ', '_')

categorical_columns = list(df.dtypes[df.dtypes == 'object'].index)

for c in categorical_columns:
    df[c] = df[c].str.lower().str.replace(' ', '_')

df.totalcharges = pd.to_numeric(df.totalcharges, errors='coerce')
df.totalcharges = df.totalcharges.fillna(0)

df.churn = (df.churn == 'yes').astype(int)

In [3]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

In [4]:
numerical = ['tenure', 'monthlycharges', 'totalcharges']

categorical = [
    'gender',
    'seniorcitizen',
    'partner',
    'dependents',
    'phoneservice',
    'multiplelines',
    'internetservice',
    'onlinesecurity',
    'onlinebackup',
    'deviceprotection',
    'techsupport',
    'streamingtv',
    'streamingmovies',
    'contract',
    'paperlessbilling',
    'paymentmethod',
]

In [12]:
def train(df_train, y_train, C=1.0):
    dicts = df_train[categorical + numerical].to_dict(orient='records')

    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)

    model = LogisticRegression(C=C, max_iter=10000)
    model.fit(X_train, y_train)
    
    return dv, model

In [13]:
def predict(df, dv, model):
    dicts = df[categorical + numerical].to_dict(orient='records')

    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]

    return y_pred

In [14]:
C = 1.0
n_splits = 5

In [15]:
kfold = KFold(n_splits=n_splits, shuffle=True, random_state=1)

scores = []

for train_idx, val_idx in kfold.split(df_full_train):
    df_train = df_full_train.iloc[train_idx]
    df_val = df_full_train.iloc[val_idx]

    y_train = df_train.churn.values
    y_val = df_val.churn.values

    dv, model = train(df_train, y_train, C=C)
    y_pred = predict(df_val, dv, model)

    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)

print('C=%s %.3f +- %.3f' % (C, np.mean(scores), np.std(scores)))

C=1.0 0.842 +- 0.007


In [16]:
scores

[0.8444081607020903,
 0.8449687955237716,
 0.8333453742725266,
 0.8347449159045853,
 0.8516755043770198]

In [17]:
dv, model = train(df_full_train, df_full_train.churn.values, C=1.0)
y_pred = predict(df_test, dv, model)

y_test = df_test.churn.values
auc = roc_auc_score(y_test, y_pred)
auc

0.8584005005037537

#### Save the model

In [19]:
import pickle

In [20]:
output_file = f"model_C={C}.bin"
output_file

'model_C=1.0.bin'

In [21]:
f_out = open(output_file, 'wb')
pickle.dump((dv, model), f_out)
f_out.close()

In [ ]:
with open(output_file, 'wb') as f_out:
    pickle.dump((dv, model), f_out)
    # do stuff

# do other stuff

#### Load the model

In [1]:
import pickle

In [2]:
model_file = 'model_C=1.0.bin'

In [3]:
with open(model_file, 'rb') as f_in:
    dv, model = pickle.load(f_in)

In [4]:
dv, model

(DictVectorizer(sparse=False), LogisticRegression(max_iter=10000))

In [5]:
customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

In [7]:
X = dv.transform([customer])

X

array([[ 1.  ,  0.  ,  0.  ,  1.  ,  0.  ,  1.  ,  0.  ,  0.  ,  1.  ,
         0.  ,  1.  ,  0.  ,  0.  , 29.85,  0.  ,  1.  ,  0.  ,  0.  ,
         0.  ,  1.  ,  1.  ,  0.  ,  0.  ,  0.  ,  1.  ,  0.  ,  1.  ,
         0.  ,  0.  ,  1.  ,  0.  ,  1.  ,  0.  ,  0.  ,  1.  ,  0.  ,
         0.  ,  1.  ,  0.  ,  0.  ,  1.  ,  0.  ,  0.  ,  1.  , 29.85]])

In [9]:
model.predict_proba(X)[0, 1]

0.6283405911692077

In [ ]:
 ls -lh model_C=1.0.bin 

### 05-03 Flask intro

A web service is a way for applications to talk to each other over HTTP - the same protocol your browser uses to load websites. One application (a client) sends a request to an address (a URL), and the other application (the server) sends back a response.

In [10]:
curl http://0.0.0.0:9696/ping

SyntaxError: invalid syntax (1153570716.py, line 1)

### 05-04 Flask deployment

In [32]:
import requests

In [39]:
url = 'http://localhost:9696/predict'

In [40]:

customer = {
    'gender': 'female',
    'seniorcitizen': 0,
    'partner': 'yes',
    'dependents': 'no',
    'phoneservice': 'no',
    'multiplelines': 'no_phone_service',
    'internetservice': 'dsl',
    'onlinesecurity': 'no',
    'onlinebackup': 'yes',
    'deviceprotection': 'no',
    'techsupport': 'no',
    'streamingtv': 'no',
    'streamingmovies': 'no',
    'contract': 'month-to-month',
    'paperlessbilling': 'yes',
    'paymentmethod': 'electronic_check',
    'tenure': 1,
    'monthlycharges': 29.85,
    'totalcharges': 29.85
}

In [41]:
requests.post(url, json=customer)

<Response [200]>

In [42]:
requests.post(url, json=customer).json()

{'churn': True, 'churn_probability': 0.6283405911692077}

In [43]:
response = requests.post(url, json=customer).json()
response

{'churn': True, 'churn_probability': 0.6283405911692077}

In [44]:
if response['churn'] == True:
    print('sending promo email to xyz-123')

sending promo email to xyz-123


 Flask warns that its development server is not suitable for production - it is single-threaded and not built for heavy load. A WSGI server is the production replacement. WSGI (Web Server Gateway Interface) is the standard way Python web applications are served: the WSGI server imports our app object and handles all the network work around it.



In [ ]:
gunicorn --bind 0.0.0.0:9696 predict:app

The last argument is module:app_object - our file is predict.py and the Flask application inside it is called app. The service behaves the same, but now it is a real production server.

### 05-05 pipenv

Instead of pip install numpy, we do pipenv install numpy

Pipfile and Pipfile.lock are created

Then, pipenv install on another computer

pipenv shell <br>
gunicorn --bind 0.0.0.0:9696 predict:app
<br>
<br>
or
<br>
<br>
pipenv run gunicorn --bind 0.0.0.0:9696 predict:app

### 05-06 Docker

docker run -it --rm python:3.8.12-slim

docker run -it --rm --entrypoint=bash python:3.8.12-slim

docker build -t zoomcamp-test .

docker run -it --rm --entrypoint= bash zoomcamp-test

docker run -it --rm -p 9696:9696 zoomcamp-test
